
## <div  style="color:blue;  font-weight:bold; font-size:100%; text-align:center;padding:12.0px; background:#ffffff">Welcome to this competition on predicting Lumbar Spine Stenosis from MR scans with AI for a quick diagnosis and Thanks for visting my Notebook! </div>

# <div  style="color:#ff0a6c; border:#0014ff solid;  font-weight:bold; font-size:120%; text-align:center;padding:12.0px; background:#000000">1. OVERVIEW</div>


<center>
<img src="https://i.postimg.cc/mZSS5gCx/32578.jpg" width=1100>
</center>

## Goal

"The goal of this competition is to create models that can be used to aid in the detection and classification of degenerative spine conditions using lumbar spine MR images. Competitors will develop models that simulate a radiologist's performance in diagnosing spine conditions."


#   Metric overview


We need to predict the **probability of stenosis** for each of the five vertebrae denoted by L1/L2,...  as well as an overall probability of any stenosis in the lumbra spine.  

"Submissions are evaluated using the average of sample **weighted log losses** and an any_severe_spinal prediction generated by the metric.""

The sample **weights** are as follows:

- "1" - for normal/mild
- "2" - for moderate
- "4" - for severe


#  Anatomical overview

<center>
<img src="https://i.postimg.cc/0jvKk9qL/maxresdefault.jpg" width=500>
</center>


<center>
<img src="https://i.postimg.cc/1tG4YbNM/2187d1a2c4675520a5969d1bf0478d8c.jpg" width=300>
</center>

<center>
<img src="https://i.postimg.cc/hGKwN5vM/63d518b29d0dc6c44ee20d955f90b59c.jpg" width=600>
</center>

# MRI overview

### MR-slices

What does **Axial** and **Sagittal** mean? Let's try to understand that:

- **Axial T2** - refers to an axial (cross-sectional) T2-weighted MRI sequence of the spine. T2-weighted images are useful for detecting pathologies like edema, inflammation, or lesions which appear hyperintense (brighter) compared to normal tissues

- **Sagittal T1** - refers to a sagittal (lengthwise) T1-weighted MRI sequence of the spine. T1-weighted images provide good anatomical detail and contrast between different **soft tissues**.

- **Sagittal T2/STIR** refers to either: A sagittal T2-weighted sequence, which is sensitive to detecting lesions, edema, and other pathologies that appear hyperintense. A sagittal STIR (**Short Tau Inversion Recovery**) sequence, which is a **fat-suppressed** T2-weighted sequence that helps highlight lesions and edema by suppressing the bright fat signal

I'll try to make it more clear the difference between the this slices:


<center>
<img src="https://i.postimg.cc/jqBkh94B/CT-Image-Planes.jpg" width=500>
</center>


### Difference between T1 and T2

<center>
<img src="https://i.postimg.cc/GmFxw3sn/Diagram-shows-the-signal-intensity-of-various-tissues-at-T1-and-T2-weighted-imaging.png" width=700>
</center>


### Real case of stenosis and MRI

<center>
<img src="https://i.postimg.cc/9fp3MjKv/7905-241447-2x.jpg" width=900>
</center>

## Dataset overview

The dataset we are using is made up of roughly 2000 MR studies. Spine radiology specialists have provided annotations to indicate the presence, vertebral level and location of any lumbar spine stenosis.

## Code requirements
This is a **code competition**, which means that submissions are made through notebooks. Furthermore, the submission notebook is subject to these conditions:
- run-time (CPU/GPU) <= 9 hours
- the submission file must be named submission.csv
- internet access disabled
- The test set is hidden, and will populated when you submit your notebook


# <div  style="color:#ff0a6c; border:#0014ff solid;  font-weight:bold; font-size:120%; text-align:center;padding:12.0px; background:#000000">2. PREPARATION</div>



In [ ]:
# packages

# standard
import numpy as np
import pandas as pd

import os
import time

# plots
import matplotlib.pyplot as plt
from matplotlib import animation, rc
import plotly.express as px
import seaborn as sns
import plotly.express as px
import cv2

import pydicom as dicom # dicom
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut

import warnings # warning handling
warnings.filterwarnings('ignore')
import glob
import json
import collections


In [ ]:
# show files
!ls -l '../input/rsna-2024-lumbar-spine-degenerative-classification'

In [ ]:
# configs
default_color_1 = 'darkblue'

In [ ]:
# read data
path = '../input/rsna-2024-lumbar-spine-degenerative-classification/'

df_train_main  = pd.read_csv(path + 'train.csv')
df_train_label = pd.read_csv(path + 'train_label_coordinates.csv')
df_train_desc  = pd.read_csv(path + 'train_series_descriptions.csv')
df_test_desc   = pd.read_csv(path + 'test_series_descriptions.csv')
df_sub         = pd.read_csv(path + 'sample_submission.csv')

# <div  style="color:#ff0a6c; border:#0014ff solid;  font-weight:bold; font-size:120%; text-align:center;padding:12.0px; background:#000000">3. DATA ANALYSIS</div>

In [ ]:
# structure of train
df_train_main.info()

In [ ]:
print("Total Cases: ", len(df_train_main))

In [ ]:
df_train_main.head(3)

Structure of train_label:

In [ ]:
df_train_label.info()

In [ ]:
df_train_label.head(3)

In [ ]:
# look at categories
for f in ['instance_number','condition','level']:
    print(df_train_label[f].value_counts())
    print('-'*50);print();

Crosstab condition vs level:

In [ ]:
pd.crosstab(df_train_label.condition, df_train_label.level)

Combine Tables:


In [ ]:
# join first two tables
df_train_step_1 = pd.merge(left=df_train_label, right=df_train_main, how='left', on='study_id').reset_index(drop=True)
df_train_step_1.head()

In [ ]:
# join with third table
df_train = pd.merge(left=df_train_step_1, right=df_train_desc, how='left', on=['study_id', 'series_id']).reset_index(drop=True)
df_train.head()

In [ ]:
# convert study_id's to categorical
df_train.study_id = df_train.study_id.astype('category')
df_train.series_id = df_train.series_id.astype('category')
df_train.head()

# Coordinates distributions

In [ ]:
# plot coordinates
default_color_1 = 'black'
sns.scatterplot(data=df_train, x='x', y='y', color=default_color_1, s = 20,  alpha=0.2)

plt.grid()
plt.show()

### Plot coordinate distributions with colored by **condition**

In [ ]:
p=['#4600c0',  '#00a419', '#111111', '#ffff00', '#ff0000']
figs_x=5  
figs_y=5
plt.figure(figsize=(figs_x, figs_y))    
sns.scatterplot(data=df_train, x='x', y='y', hue='condition',  palette=p, s = 20,  alpha=0.2)
plt.legend(bbox_to_anchor=(1.2,1), loc=2, prop={'size': 10}, markerscale = 2, framealpha=0.8, facecolor='white')
plt.grid()
plt.xlim([0, df_train.x.max()])
plt.ylim([0, df_train.y.max()])
plt.show()


### Plot xy-Density of cases in stenosis split

In [ ]:
numerical_features =   ['Left Neural Foraminal Narrowing', 'Right Neural Foraminal Narrowing',
                        'Left Subarticular Stenosis',      'Right Subarticular Stenosis',
                        'Spinal Canal Stenosis']
gs=150
n=2 # num of columns
a=0 
k=1;
colorlabels = 'darkblue'

Label_size = 10 # Size font of xy labels
Title_size = 20 # Size font of Title
figs_x=13   
figs_y=5
plt.figure(figsize=(figs_x, figs_y))    
for i in numerical_features:
        
        d = df_train[df_train.condition == i]
        plt.subplot(1,n, k)
        #sns.jointplot(data=d, x='x', y='y', color='white', alpha=0.25)
        plt.hexbin(data=d, x='x', y='y',  gridsize=gs, cmap='CMRmap', bins='log', alpha = 1)
        
        #plt.colorbar(label='count in bin')
        plt.colorbar().set_label(label='count in bin',size=10, color  = 'grey')
        #plt.colorbar(size=8)
        
        #plt.xlabel(f'{i}')
        #plt.ylabel(f'{j}')
        plt.tick_params(axis='x', labelsize=Label_size)
        plt.tick_params(axis='y', labelsize=Label_size)
        
        plt.xlim([0, df_train.x.max()])
        plt.ylim([0, df_train.y.max()])
        plt.xlabel(f'x', fontsize=Label_size, color = colorlabels)
        plt.ylabel(f'y', fontsize=Label_size, color = colorlabels) 
        
        plt.title(f'{i}', color='black', fontsize=Title_size)
        k=k+1
        if k == (n+1):    
            k=1
            plt.show()
            plt.figure(figsize=(figs_x, figs_y)) 


### Plot coordinates distributions with colored by **level**

In [ ]:
p=['#4600c0',  '#00a419', '#111111', '#ffff00', '#ff0000']
figs_x=5
figs_y=5
plt.figure(figsize=(figs_x, figs_y))    
sns.scatterplot(data=df_train, x='x', y='y', hue='level',  palette=p, s = 20,  alpha=0.3)
plt.legend(bbox_to_anchor=(1.2,1), loc=2, prop={'size': 10}, markerscale = 2, framealpha=0.8, facecolor='white', reverse=True)
plt.grid()
plt.xlim([0, df_train.x.max()])
plt.ylim([0, df_train.y.max()])
plt.show()

In [ ]:
df_train.level.unique()
df_train.head(3)

In [ ]:
numerical_features =   df_train.level.unique() #L1/L2', 'L2/L3', 'L3/L4', 'L4/L5', 'L5/S1'
gs=150
n=2 # num of columns
a=0 
k=1;
colorlabels = 'darkblue'

Label_size = 10 # Size font of xy labels
Title_size = 20 # Size font of Title
figs_x=13   
figs_y=5
plt.figure(figsize=(figs_x, figs_y))    
plt.suptitle("xy-Density of cases in L-split", fontsize=Title_size+2, fontweight='bold', y=1.015)
for i in numerical_features:
        
        #d = df_train[ (df_train.level == i) & (df_train.condition == 'Left Neural Foraminal Narrowing') ]
        d = df_train[ (df_train.level == i) ]
        plt.subplot(1,n, k)
        #sns.jointplot(data=d, x='x', y='y', color='white', alpha=0.25)
        plt.hexbin(data=d, x='x', y='y',  gridsize=gs, cmap='CMRmap', bins='log', alpha = 1)
        
        #plt.colorbar(label='count in bin')
        plt.colorbar().set_label(label='count in bin',size=10, color  = 'grey')
        #plt.colorbar(size=8)
        
        #plt.xlabel(f'{i}')
        #plt.ylabel(f'{j}')
        plt.tick_params(axis='x', labelsize=Label_size)
        plt.tick_params(axis='y', labelsize=Label_size)
        
        plt.xlim([0, df_train.x.max()])
        plt.ylim([0, df_train.y.max()])
        plt.xlabel(f'x', fontsize=Label_size, color = colorlabels)
        plt.ylabel(f'y', fontsize=Label_size, color = colorlabels) 
        
        plt.title(f'{i}', color='black', fontsize=Title_size)
        k=k+1
        if k == (n+1):    
            k=1
            plt.show()
            plt.figure(figsize=(figs_x, figs_y)) 
            

### Plot stenosis distributions with colored by **MR-split**

In [ ]:
p=['#000000', '#4600c0', '#ff0000']
figs_x=5  
figs_y=5
plt.figure(figsize=(figs_x, figs_y))    
sns.scatterplot(data=df_train, x='x', y='y', hue='series_description',  palette=p, s = 20,  alpha=0.4)
plt.legend(bbox_to_anchor=(1.2,1), loc=2, prop={'size': 10}, markerscale = 2, framealpha=0.8, facecolor='white')
plt.grid()
plt.xlim([0, df_train.x.max()])
plt.ylim([0, df_train.y.max()])
plt.show()


In [ ]:
numerical_features =   df_train.series_description.unique()
gs=200
n=2 # num of columns
a=0 
k=1;
colorlabels = 'darkblue'

Label_size = 10 # Size font of xy labels
Title_size = 20 # Size font of Title
figs_x=13   
figs_y=5
plt.figure(figsize=(figs_x, figs_y))    
plt.suptitle("xy-Density of cases in plane-split", fontsize=Title_size+2, fontweight='bold', y=1.015)
for i in numerical_features:
        
        d = df_train[df_train.series_description == i]
        plt.subplot(1,n, k)
        #sns.jointplot(data=d, x='x', y='y', color='white', alpha=0.25)
        plt.hexbin(data=d, x='x', y='y',  gridsize=gs, cmap='CMRmap', bins='log', alpha = 1)
        
        #plt.colorbar(label='count in bin')
        plt.colorbar().set_label(label='count in bin',size=10, color  = 'grey')
        #plt.colorbar(size=8)
        
        plt.tick_params(axis='x', labelsize=Label_size)
        plt.tick_params(axis='y', labelsize=Label_size)
        
        plt.xlim([0, df_train.x.max()])
        plt.ylim([0, df_train.y.max()])
        plt.xlabel(f'x', fontsize=Label_size, color = colorlabels)
        plt.ylabel(f'y', fontsize=Label_size, color = colorlabels) 
        
        plt.title(f'{i}', color='black', fontsize=Title_size)
        k=k+1
        if k == (n+1):    
            k=1
            plt.show()
            plt.figure(figsize=(figs_x, figs_y)) 

# Target distribution

### Let's exploring target distribution in different splits

At the first, creating the new column by concatenating values from columns entiteld **conditions** and **level**

In [ ]:
train_label_df = df_train_label.copy()
train_data_df = df_train_main.copy()
train_label_df['new_col'] = df_train_label['condition'].str.lower().str.replace(' ', '_') +  '_' + df_train_label['level'].str.lower().str.replace('/', '_')

# Step 2: Merge the values from train_data_df based on study_id and the newly created column names
def get_target_value(row):
    study_id = row['study_id']
    new_col = row['new_col']
    return train_data_df[train_data_df['study_id'] == study_id][new_col].values[0]

train_label_df['target'] = train_label_df.apply(get_target_value, axis=1)

# Drop the 'new_col' column 
train_label_df.drop(columns=['new_col'], inplace=True)

# Copy the train_label_df to new DataFrame
final_train_df = train_label_df.copy()

# Drop the 'new_col' column 
train_label_df.drop(columns=['target'], inplace=True)

#final_train_df.to_csv('final_train.csv')
final_train_df.head()

Plotting the distribution of the target variable:

In [ ]:
#plt.suptitle("Target class distribution \n (1 - default, 0 - not default)", fontsize=15, fontweight='bold', y=1.0)

name = ['Normal/Mild', 'Moderate', 'Severe']
p=['#d0d0d0', '#ffba07', '#ff0000']
plt.pie(final_train_df.target.value_counts(normalize=True), autopct = '%1.f%%', 
colors = sns.color_palette(p), startangle=90, wedgeprops=dict(width=0.25), labeldistance=1.2, 
counterclock=False, radius=1)
plt.title(f'Total', color='blue', fontsize=Title_size)
plt.legend(name, loc='lower left',  prop={'size': 12}, markerscale = 3, framealpha=0.8, facecolor='white')


Plotting the distribution of the target variable in L/L splits:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
colours = {"male": "#273c75", "female": "#44bd32"}
for idx, d in enumerate(['foraminal', 'subarticular', 'canal']):
    diagnosis = list(filter(lambda x: x.find(d) > -1, df_train_main.columns))
    dff = df_train_main[diagnosis]
    with warnings.catch_warnings():
        warnings.simplefilter(action="ignore", category=FutureWarning)
        value_counts = dff.apply(pd.value_counts).fillna(0).T
      
    value_counts.plot(kind='bar', stacked=True, ax=axes[idx], cmap='Set1_r')
    
    axes[idx].tick_params(axis='x', labelsize=16) 
    
    axes[idx].set_title(f"{d} distribution",  fontsize=28)

    
#color=plotdata['gender'].replace(colours)    

Other view on target variable in L/L splits:

In [ ]:
n=3 # num of columns
a=0 
k=1;
colorlabels = 'darkblue'

p=['#d0d0d0', '#ffba07', '#ff0000']
label = df_train_main.columns.drop('study_id').tolist()        
        
Label_size = 12 # Size font of xy labels
Title_size = 15 # Size font of Title
figs_x=16
figs_y=5
plt.figure(figsize=(figs_x, figs_y))    
for i in label:
    plt.subplot(1, n, k)
    plt.pie(df_train_main[i].value_counts(normalize=True), autopct = '%1.f%%', 
        colors = sns.color_palette(p), startangle=90, wedgeprops=dict(width=0.25), 
        counterclock=False, radius=1)
    plt.title(f'{i}', color='blue', fontsize=Title_size)
    plt.legend(name, loc='lower left',  prop={'size': 12}, markerscale = 3, framealpha=0.8, facecolor='white')
    k=k+1
    if k == (n+1):    
        k=1
        plt.show()
        plt.figure(figsize=(figs_x, figs_y))

Let's pick a study to see images

In [ ]:
df_train.head(3)

What do we need to predict?¶
For a given study in the test data, we need to predict the severity condition of all types of stenosis at all levels. So from a first impression it seems a very heavy problem, because we need to predict for each of the 25 different possible combinations.

# <div  style="color:#ff0a6c;  border:#0014ff solid; font-weight:bold; font-size:120%; text-align:center;padding:12.0px; background:#000000">4. MR-IMAGES</div>

# What is DICOM?

A .dcm file follows the **Digital Imaging and Communications in Medicine** (DICOM) format. It is the standard format used for storing medical images and related metadata. It dates back to 1983, although it has been revised many times.

We can use the pydicom library to open and explore these files.

In [ ]:
import pydicom as dicom
import matplotlib.patches as patches

In [ ]:
train_label_coordinates=df_train_label

In [ ]:
train_label_coordinates[train_label_coordinates.study_id==4003253]

## Introduction to image visualisation with **imshow()**


Let's create random array 100x100:

In [ ]:
rand_data = np.random.randint(0, 255, (100, 100))
rand_data

Visualizing random array with three color map:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3))

axes[0].imshow(rand_data, cmap ='viridis')
axes[1].imshow(rand_data, cmap ='CMRmap')
axes[2].imshow(rand_data, cmap ='binary')

plt.show()

# Visualizing MR-images

In [ ]:
train_label_coordinates['series_description'] = df_train.series_description

In [ ]:
def mrt(id, ser, inst):
    lag=20
    path2 = path+'train_images/' + str(id) +'/' + str(ser)+'/' + str(inst) + '.dcm'

    ds = dicom.dcmread(path2)
    fig, ax = plt.subplots(figsize=(16, 8))
    from matplotlib.colors import LogNorm 

    ax.imshow(ds.pixel_array, cmap ='CMRmap')     # Display the image

    # Create a legend
    legend_elements = []

    # Plot the coordinates for the current condition
    ab = train_label_coordinates[(train_label_coordinates.study_id==id) & 
                                          (train_label_coordinates.instance_number==inst)&
                                         (train_label_coordinates.series_id==ser)]

    a = 25 * max(ds.pixel_array.shape)/640
    for _, row in ab.iterrows():
        x, y = row['x'], row['y']

        rect2 = patches.Rectangle((x - a, y - a), 2*a, 2*a, linewidth=2, edgecolor='white', facecolor='none')
        rect1 = patches.Rectangle((x - a, y - a), 2*a, 2*a, linewidth=2, facecolor='white', alpha = 0.25)

        ax.add_patch(rect2)
        ax.add_patch(rect1)

        # Add the condition to the legend
        legend_elements.append(patches.Patch(facecolor='none', edgecolor='r', ))

    # Add title
    title = f"{ab.series_description.unique()}, Study: {id}, Series: {ser}, Instance: {inst}"
    ax.set_title(title, fontsize=20)

    # Display additional columns:
    for _, row in ab.iterrows():
        text = f"level {row['level']}, {row['condition']}"
        ax.text(row['x'] + lag, row['y']+np.random.randint(-15, 15), text, fontsize=10, color='white', verticalalignment='center_baseline')
    
    plt.show() 

In [ ]:
#Case #1
id = 4003253
ser= 702807833
inst=8

mrt(id, ser, inst)

In [ ]:
#Case #2
id = 4290709089
ser= 4237840455
inst=11

mrt(id, ser, inst)

In [ ]:
#Case #3
id = 4003253
ser= 1054713880
inst=4

mrt(id, ser, inst)

In [ ]:
#Case #4
id = 4003253
ser= 2448190387
inst=11

mrt(id, ser, inst)

# 3D MR-slides visualisation

In [ ]:
def MR3d(id, ser):
    
    path_to_folder = path + "train_images/"+str(id)+'/'+str(ser)
    def load_dicom(path):
        dicom = pydicom.read_file(path)
        data = dicom.pixel_array
        data = data - np.min(data)
        if np.max(data) != 0:
            data = data / np.max(data)
        data = (data * 255).astype(np.uint8)
        return data

    rc('animation', html='jshtml')

    def load_dicom(filename):
        ds = pydicom.dcmread(filename)
        return ds.pixel_array

    def load_dicom_line(path):
        t_paths = sorted(
            glob.glob(os.path.join(path, "*")), 
            key=lambda x: int(os.path.splitext(os.path.basename(x))[0].split("-")[-1]),
        )
        images = []
        for filename in t_paths:
            data = load_dicom(filename)
            if data.max() == 0:
                continue
            images.append(data)
        return images

    def create_animation(ims):
        fig = plt.figure(figsize=(6, 6))
        plt.axis('off')
        im = plt.imshow(ims[0], cmap="CMRmap")
        text = plt.text(0.05, 0.05, f'Slide {1}', transform=fig.transFigure, fontsize=16, color='darkblue')

        def animate_func(i):
            im.set_array(ims[i])
            text.set_text(f'Slide {i+1}')  
            return [im]
        plt.title(f'id = {id}, series = {ser}')
        
        plt.close()  

        return animation.FuncAnimation(fig, animate_func, frames=len(ims), interval=1000//10) #24

    images = load_dicom_line(path_to_folder)
    
    return create_animation(images)



In [ ]:
#Case #1
id = 4003253
ser= 702807833

MR3d(id, ser)

In [ ]:
#Case #2
id = 4290709089
ser= 4237840455

MR3d(id, ser)

In [ ]:
#Case #3
id = 4003253
ser= 1054713880

MR3d(id, ser)

In [ ]:
#Case #4
id = 4003253
ser= 2448190387

MR3d(id, ser)

# <div  style="color:#ff0a6c;  border:#0014ff solid; font-weight:bold; font-size:120%; text-align:center;padding:12.0px; background:#000000">5. MODELLING</div>

Define Competition metrics (for future):

In [ ]:
import pandas.api.types
import sklearn.metrics


class ParticipantVisibleError(Exception):
    pass


def get_condition(full_location: str) -> str:
    # Given an input like spinal_canal_stenosis_l1_l2 extracts 'spinal'
    for injury_condition in ['spinal', 'foraminal', 'subarticular']:
        if injury_condition in full_location:
            return injury_condition
    raise ValueError(f'condition not found in {full_location}')
    
    
def score(
        solution: pd.DataFrame,
        submission: pd.DataFrame,
        row_id_column_name: str,
        any_severe_scalar: float
    ) -> float:
    '''
    Pseudocode:
    1. Calculate the sample weighted log loss for each medical condition:
    2. Derive a new any_severe label.
    3. Calculate the sample weighted log loss for the new any_severe label.
    4. Return the average of all of the label group log losses as the final score, normalized for the number of columns in each group.
       This mitigates the impact of spinal stenosis having only half as many columns as the other two conditions.
    '''

    target_levels = ['normal_mild', 'moderate', 'severe']

    # Run basic QC checks on the inputs
    if not pandas.api.types.is_numeric_dtype(submission[target_levels].values):
        raise ParticipantVisibleError('All submission values must be numeric')

    if not np.isfinite(submission[target_levels].values).all():
        raise ParticipantVisibleError('All submission values must be finite')

    if solution[target_levels].min().min() < 0:
        raise ParticipantVisibleError('All labels must be at least zero')
    if submission[target_levels].min().min() < 0:
        raise ParticipantVisibleError('All predictions must be at least zero')

    solution['study_id']  = solution['row_id'].apply(lambda x: x.split('_')[0])
    solution['location']  = solution['row_id'].apply(lambda x: '_'.join(x.split('_')[1:]))
    solution['condition'] = solution['row_id'].apply(get_condition)

    del solution[row_id_column_name]
    del submission[row_id_column_name]
    assert sorted(submission.columns) == sorted(target_levels)

    submission['study_id']  = solution['study_id']
    submission['location']  = solution['location']
    submission['condition'] = solution['condition']

    condition_losses  = []
    condition_weights = []
    for condition in ['spinal', 'foraminal', 'subarticular']:
        condition_indices = solution.loc[solution['condition'] == condition].index.values
        condition_loss = sklearn.metrics.log_loss(
            y_true=solution.loc[condition_indices, target_levels].values,
            y_pred=submission.loc[condition_indices, target_levels].values,
            sample_weight=solution.loc[condition_indices, 'sample_weight'].values
        )
        condition_losses.append(condition_loss)
        condition_weights.append(1 / solution.loc[condition_indices, 'location'].nunique())

        
    any_severe_spinal_labels      = pd.Series(solution.loc[solution    ['condition'] == 'spinal'].groupby('study_id')['severe'].max())
    any_severe_spinal_weights     = pd.Series(solution.loc[solution    ['condition'] == 'spinal'].groupby('study_id')['sample_weight'].max())
    any_severe_spinal_predictions = pd.Series(submission.loc[submission['condition'] == 'spinal'].groupby('study_id')['severe'].max())
    any_severe_spinal_loss = sklearn.metrics.log_loss(
        y_true=any_severe_spinal_labels,
        y_pred=any_severe_spinal_predictions,
        sample_weight=any_severe_spinal_weights
    )
    condition_losses.append(any_severe_spinal_loss)
    condition_weights.append(any_severe_scalar)
    return np.average(condition_losses, weights=condition_weights)

# Model #1 Simple frequencies

In [ ]:
df_test_desc

In [ ]:
df_sub.head(5)

In [ ]:
df_train_main.head(3)

In [ ]:
df_unpivoted = df_train_main.melt(id_vars='study_id', var_name='condition', value_name='status')
df_unpivoted.head()

In [ ]:
#df_unpivoted = df_train_main.melt(id_vars='study_id', var_name='condition', value_name='status')
df_unpivoted[df_unpivoted.study_id ==4003253]

In [ ]:
frequency_table = df_unpivoted.groupby('condition')['status'].value_counts(normalize=True).unstack(fill_value=0)
frequency_table = frequency_table.reset_index()
frequency_table.rename(columns={ 'Normal/Mild': 'normal_mild', 'Moderate': 'moderate', 'Severe': 'severe'}, inplace=True)
frequency_table.round(2)

In [ ]:
df_sub['condition'] = df_sub['row_id'].str.extract(r'_(.*)')
df_sub.round(2)

In [ ]:
# freq's of all stenosis
df_sub = pd.merge(df_sub[['row_id', 'condition']],frequency_table, on='condition', how='inner')[['row_id', 'normal_mild', 'moderate', 'severe']]
df_sub.round(2)

In [ ]:
df_train

In [ ]:
# setup baseline model just using observed frequencies

# Define the path to  test images directory
test_images_dir = "/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/test_images"


# Get all unique IDs from the filenames in the test images directory
test_ids = [filename.split('.')[0] for filename in os.listdir(test_images_dir)]
test_ids =[4003252, 4003253] #temporary, for debugging
unique_ids = list(set(test_ids))

# Generate the row_ids needed for submission by repeating each unique_id for each condition from df_train_main
conditions = ['left_neural_foraminal_narrowing_l1_l2',
              'left_neural_foraminal_narrowing_l2_l3',
              'left_neural_foraminal_narrowing_l3_l4',
              'left_neural_foraminal_narrowing_l4_l5',
              'left_neural_foraminal_narrowing_l5_s1',
              'left_subarticular_stenosis_l1_l2',
              'left_subarticular_stenosis_l2_l3',
              'left_subarticular_stenosis_l3_l4',
              'left_subarticular_stenosis_l4_l5',
              'left_subarticular_stenosis_l5_s1',
              'right_neural_foraminal_narrowing_l1_l2',
              'right_neural_foraminal_narrowing_l2_l3',
              'right_neural_foraminal_narrowing_l3_l4',
              'right_neural_foraminal_narrowing_l4_l5',
              'right_neural_foraminal_narrowing_l5_s1',
              'right_subarticular_stenosis_l1_l2',
              'right_subarticular_stenosis_l2_l3',
              'right_subarticular_stenosis_l3_l4',
              'right_subarticular_stenosis_l4_l5',
              'right_subarticular_stenosis_l5_s1',
              'spinal_canal_stenosis_l1_l2',
              'spinal_canal_stenosis_l2_l3',
              'spinal_canal_stenosis_l3_l4',
              'spinal_canal_stenosis_l4_l5',
              'spinal_canal_stenosis_l5_s1']

row_ids = [f"{id}_{condition}" for id in unique_ids for condition in conditions]

# Create DataFrame
df_submission = pd.DataFrame(row_ids, columns=['row_id'])
# df_submission['normal_mild'] = 0.333333
# df_submission['moderate']    = 0.333333
# df_submission['severe']      = 0.333333

df_submission['normal_mild'] = 0.4242
df_submission['moderate']    = 0.3031
df_submission['severe']      = 0.2727


# df_submission['normal_mild'] = 0.81
# df_submission['moderate'] = 0.14
# df_submission['severe'] = 0.05
#df_submission

# Model #2 XGBoost without tuning

In [ ]:
from sklearn.preprocessing import LabelEncoder
test_series = pd.read_csv('/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/test_series_descriptions.csv')
df_train = pd.read_csv('/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/train.csv')

df_train_melted = df_train.melt(id_vars=['study_id'], var_name='condition_level', value_name='severity')
df_train_melted[['condition', 'level']] = df_train_melted['condition_level'].str.rsplit('_', n=2, expand=True).iloc[:, 1:]

le_severity = LabelEncoder()
df_train_melted['severity_encoded'] = le_severity.fit_transform(df_train_melted['severity'])

X_train = df_train_melted[['study_id', 'condition', 'level']]
y_train = df_train_melted['severity_encoded']

X_train = pd.get_dummies(X_train, columns=['condition', 'level'])
test_rows = []
for _, row in test_series.iterrows():
    for condition in ['left_neural_foraminal_narrowing', 'right_neural_foraminal_narrowing', 'left_subarticular_stenosis', 'right_subarticular_stenosis', 'spinal_canal_stenosis']:
        for level in ['l1_l2', 'l2_l3', 'l3_l4', 'l4_l5', 'l5_s1']:
            test_rows.append({
                'study_id': row['study_id'],
                'condition': condition,
                'level': level
            })

X_test = pd.DataFrame(test_rows)
X_test = pd.get_dummies(X_test, columns=['condition', 'level'])
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [ ]:
X_train

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(random_state=1)
model.fit(X_train, y_train)

In [ ]:
predictions_proba = model.predict_proba(X_test)
predictions_df = pd.DataFrame(predictions_proba, columns=le_severity.classes_)
predictions_df['study_id'] = X_test['study_id'].values
predictions_df['condition_level'] = X_test.index.map(lambda idx: f"{test_rows[idx]['condition']}_{test_rows[idx]['level']}")
predictions_df['row_id'] = predictions_df['study_id'].astype(str) + '_' + predictions_df['condition_level']

# <div  style="color:#ff0a6c;  border:#0014ff solid; font-weight:bold; font-size:120%; text-align:center;padding:12.0px; background:#000000">6. SUBMISSION</div>

In [ ]:
df_sub1 = pd.read_csv('/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/sample_submission.csv')

In [ ]:
# Save the DataFrame to a CSV file for submission
df_submission.to_csv('submission.csv', index=False)

In [ ]:
normal_mild_value = predictions_df['Normal/Mild'].iloc[0]
moderate_value = predictions_df['Moderate'].iloc[0]
severe_value = predictions_df['Severe'].iloc[0]


df_sub['normal_mild'] = normal_mild_value/2
df_sub['moderate'] =moderate_value*2
df_sub['severe'] = 1-(normal_mild_value/2 +moderate_value*2)- severe_value


normal_mild_value = 0.36
moderate_value = 0.426666667
severe_value = 0.213333333

df_sub['normal_mild'] = normal_mild_value
df_sub['moderate']    = moderate_value
df_sub['severe']      = severe_value


In [ ]:
df_sub.sample(4)

In [ ]:
df_sub.to_csv('submission.csv', index=False)

## <div  style="color:blue;   font-size:120%; text-align:center;padding:12.0px; background:#ffffff">Thanks for viewing my work.   If you like it give feedback to improve the notebook.     Have a beautiful day my friend! </div>